# NYC taxi fleet rebalancing — revised, leakage-safe experiment

This notebook replaces the original 52-cell prototype with a compact end-to-end experiment. It keeps the useful high-level idea—a shared graph policy, centralized critic, and historical TLC trip flows—but corrects the simulator, policy likelihood, data split, and evaluation.

## Audit of the original notebook

| Area | Original behavior | Assessment | Replacement |
|---|---|---|---|
| Demand | Divided pickup counts by each zone's monthly maximum, then treated the fractions as requests | **Invalid simulation scale** | Integer OD request counts |
| Forecast | Averaged the next three realized bins | **Direct future leakage** | Frozen train-only weekday × time-of-day profile |
| Split | Declared train/test constants but sampled resets from all 31 days | **No held-out test** | Chronological 8-month train / 2-month validation / 2-month locked test |
| Route/fare estimates | Fit on the full month | **Test leakage** | Fit on training months only with robust shrinkage/fallback |
| Graph | Shapefile path failed; any observed OD pair became an adjacent edge; random truncation to 10 | **Wrong spatial graph** | Polygon-touch graph plus symmetric nearest-centroid links; retain every neighbor |
| Relocation | Every idle driver moved; no stay action; in-place loop let drivers cascade through zones | **Order-dependent, nonphysical** | Stay action; simultaneous integer flows; delayed arrivals |
| Passenger trips | Used intra-zone fare, completed every trip instantly, returned drivers to pickup origins | **No OD or duration dynamics** | OD fare and duration; later arrival at recorded destination |
| Accounting | Inconsistent driver state; relocation cost divided twice; matched count equaled revenue | **Incorrect metrics** | Conservation assertions and unit-consistent accounting |
| GCN | Unnormalized adjacency without self-loops; unused sparse tensor | **Unstable and redundant** | Symmetric normalization with self-loops and batched PyTorch operations |
| PPO | Unsampled Gaussian over simplex values, followed by another softmax | **Wrong likelihood and no exploration** | Gaussian over unconstrained relocation logits; one masked softmax in environment |
| PPO ratio | One joint log probability over all zones/actions | **Numerically brittle** | Per-zone factorized ratios with shared cooperative advantage |
| Targets | Terminal bootstrap from last pre-action state; no clipped value loss | **Biased/incomplete PPO** | Next-state values, done masks, GAE, policy and value clipping |
| Evaluation | One reset from training environment; no baselines or uncertainty | **Not an experiment** | Locked days, baselines, paired intervals, multi-seed plan |

TLC records completed trips, not rejected requests or request timestamps. Therefore missed demand means historically observed trips the simulated fleet could not serve; it is not total latent demand. Passenger wait time is intentionally not reported.


## Predeclared experiment and decision log

**Focal question.** On chronologically held-out 2023 months, does a graph-parameter-shared MAPPO policy improve simulated operating value relative to transparent non-learning policies while conserving vehicles and respecting trip/relocation delay?

**Primary outcome.** Gross expected receipts minus empty-relocation cost minus a penalty for missed observed requests. The coefficients are assumptions, not estimated causal quantities, and require sensitivity analysis.

**Secondary outcomes.** Observed-trip service rate, gross expected receipts, missed observed requests, paid-driver utilization, empty-relocation utilization, and empty driver-minutes.

**Split and replication.**

- Train January–August; validation September–October; locked test November–December.
- Full run uses five training seeds. Quick mode uses one and is only a pipeline check.
- Compare methods on identical test days using paired day-block bootstrap intervals.
- Inspect test results after choices are frozen. Repeat across months/seasons for publishable inference.

| ID | Candidate idea | Information gain | Rigor | Feasibility | Adversarial concern | Decision |
|---|---|---:|---:|---:|---|---|
| I1 | Repair simulator before network tuning | High | High | High | More computation | **Required gate** |
| I2 | Shared GCN actor + centralized critic MAPPO | Medium | Medium | Medium | May not beat simple demand tracking | Retain as candidate |
| I3 | Train-only seasonal forecast | High | High | High | Weak with three weeks | Retain; later ablate |
| I4 | Rolling mixed-integer allocation baseline | High | High | Medium/low | Short horizon | Optional full-run baseline |
| I5 | Port all preprocessing to GPU | Low | Medium | Low | Setup/transfers may dominate | Reject |

**Disconfirmation.** Do not prefer MAPPO if conservation fails, locked primary outcome is worse than baselines, estimates are unstable across seeds, or gains disappear under plausible cost sensitivity. This is an exploratory simulation, not a deployment claim.


### 1. Install dependencies



In [ ]:
# Install only direct dependencies. Prefer a lockfile in managed environments.
%pip install -q "duckdb>=1.0" "pyarrow>=16" "geopandas>=0.14" \
    "shapely>=2.0" "gymnasium>=0.29" "scipy>=1.11" \
    "pandas>=2.0" "matplotlib>=3.8" "requests>=2.31"


### 2. Imports, configuration, and reproducibility



In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Iterable

import duckdb
import geopandas as gpd
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces
from scipy import sparse
from scipy.optimize import Bounds, LinearConstraint, milp


@dataclass(frozen=True)
class Config:
    seed: int = 42
    year: int = 2023
    months: tuple[int, ...] = tuple(range(1, 13))
    n_zones: int = 263
    bin_minutes: int = 30
    fleet_size: int = 3_000
    train_days: int = 243  # January--August
    val_days: int = 61     # September--October
    test_days: int = 61    # November--December
    nearest_neighbors: int = 3
    fare_prior_strength: float = 25.0
    relocation_cost_per_minute: float = 0.35
    missed_request_penalty: float = 10.0
    service_rate_reward: float = 25.0
    max_relocation_fraction: float = 0.25
    stay_logit_bias: float = 2.5
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_eps: float = 0.20
    entropy_coef: float = 0.01
    value_coef: float = 0.50
    actor_lr: float = 3e-4
    critic_lr: float = 7e-4
    grad_clip: float = 0.5
    ppo_epochs: int = 6
    minibatch_steps: int = 128
    hidden_dim: int = 96
    quick_mode: bool = False

    @property
    def bins_per_day(self) -> int:
        return 24 * 60 // self.bin_minutes

    @property
    def total_days(self) -> int:
        return self.train_days + self.val_days + self.test_days

    @property
    def total_bins(self) -> int:
        return self.total_days * self.bins_per_day

    @property
    def train_end(self) -> int:
        return self.train_days * self.bins_per_day

    @property
    def val_end(self) -> int:
        return (self.train_days + self.val_days) * self.bins_per_day

    @property
    def rollout_days(self) -> int:
        return 2 if self.quick_mode else 8

    @property
    def max_updates(self) -> int:
        return 8 if self.quick_mode else 250

    @property
    def validation_every(self) -> int:
        return 2 if self.quick_mode else 10

    @property
    def early_stop_checks(self) -> int:
        return 4 if self.quick_mode else 10

    @property
    def training_seeds(self) -> tuple[int, ...]:
        return (11,) if self.quick_mode else (11, 29, 47, 71, 101)


CFG = Config()
DATA_DIR = Path("/content/tlc_rl_revised") if Path("/content").exists() else Path.cwd() / "tlc_rl_revised"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRIP_FILES = [
    DATA_DIR / f"yellow_tripdata_{CFG.year}-{month:02d}.parquet"
    for month in CFG.months
]
TRIP_GLOB = (DATA_DIR / f"yellow_tripdata_{CFG.year}-*.parquet").as_posix()
ZONE_ZIP = DATA_DIR / "taxi_zones.zip"
ZONE_DIR = DATA_DIR / "taxi_zones"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2))
print(f"device={DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(DEVICE)
    print(f"gpu={props.name}; memory={props.total_memory / 2**30:.1f} GiB")


### 3. Download full-year TLC data and zone geometry



In [ ]:
TRIP_URLS = {
    path: (
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/"
        f"{path.name}"
    )
    for path in TRIP_FILES
}
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"


def download(url: str, destination: Path, chunk_bytes: int = 1 << 20) -> None:
    if destination.exists() and destination.stat().st_size > 0:
        return
    tmp = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tmp.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=chunk_bytes):
                if chunk:
                    handle.write(chunk)
    tmp.replace(destination)


for trip_file, trip_url in TRIP_URLS.items():
    download(trip_url, trip_file)
download(ZONE_URL, ZONE_ZIP)
if not ZONE_DIR.exists():
    ZONE_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ZONE_ZIP) as archive:
        archive.extractall(ZONE_DIR)

shape_candidates = list(ZONE_DIR.rglob("taxi_zones.shp"))
if len(shape_candidates) != 1:
    raise FileNotFoundError(f"Expected one taxi_zones.shp, found {shape_candidates}")
SHAPEFILE = shape_candidates[0]
print(
    f"trip data: {len(TRIP_FILES)} months, "
    f"{sum(path.stat().st_size for path in TRIP_FILES) / 2**30:.2f} GiB"
)
print(f"zones: {SHAPEFILE}")


### 4. Build full-year integer OD demand and train-only route statistics



In [ ]:
# DuckDB scans Parquet on CPU without loading the raw table into Python.
con = duckdb.connect()
start = f"{CFG.year}-01-01 00:00:00"
next_year = f"{CFG.year + 1}-01-01 00:00:00"
clean_cte = f'''
WITH clean AS (
    SELECT
        CAST(FLOOR(DATEDIFF('minute', TIMESTAMP '{start}', tpep_pickup_datetime)
             / {CFG.bin_minutes}) AS INTEGER) AS time_bin,
        CAST(PULocationID - 1 AS INTEGER) AS pu,
        CAST(DOLocationID - 1 AS INTEGER) AS dropoff_zone,
        DATEDIFF('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS duration_min,
        CAST(fare_amount + COALESCE(tip_amount, 0) AS DOUBLE) AS receipts
    FROM read_parquet('{TRIP_GLOB}')
    WHERE tpep_pickup_datetime >= TIMESTAMP '{start}'
      AND tpep_pickup_datetime < TIMESTAMP '{next_year}'
      AND tpep_dropoff_datetime >= tpep_pickup_datetime
      AND PULocationID BETWEEN 1 AND {CFG.n_zones}
      AND DOLocationID BETWEEN 1 AND {CFG.n_zones}
      AND COALESCE(passenger_count, 1) >= 1
)
'''
od_counts = con.execute(clean_cte + f'''
SELECT time_bin, pu, dropoff_zone, COUNT(*)::INTEGER AS requests
FROM clean
WHERE time_bin BETWEEN 0 AND {CFG.total_bins - 1}
  AND duration_min BETWEEN 1 AND 120
  AND receipts BETWEEN 2 AND 300
GROUP BY time_bin, pu, dropoff_zone
ORDER BY time_bin, pu, dropoff_zone
''').df()
train_routes = con.execute(clean_cte + f'''
SELECT pu, dropoff_zone, COUNT(*)::INTEGER AS n,
       MEDIAN(duration_min)::DOUBLE AS duration_min,
       MEDIAN(receipts)::DOUBLE AS receipts
FROM clean
WHERE time_bin BETWEEN 0 AND {CFG.train_end - 1}
  AND duration_min BETWEEN 1 AND 120
  AND receipts BETWEEN 2 AND 300
GROUP BY pu, dropoff_zone
''').df()
con.close()

idx_t = torch.as_tensor(od_counts.time_bin.to_numpy(), dtype=torch.long)
idx_i = torch.as_tensor(od_counts.pu.to_numpy(), dtype=torch.long)
idx_j = torch.as_tensor(od_counts["dropoff_zone"].to_numpy(), dtype=torch.long)
vals = torch.as_tensor(od_counts.requests.to_numpy(), dtype=torch.int16)
demand_od = torch.zeros((CFG.total_bins, CFG.n_zones, CFG.n_zones), dtype=torch.int16)
demand_od.index_put_((idx_t, idx_i, idx_j), vals, accumulate=True)
if int(demand_od.max()) > torch.iinfo(torch.int16).max:
    raise ValueError("A time-bin OD count exceeds int16 capacity; use int32 storage.")
demand_zone = demand_od.sum(dim=2).to(torch.float32)
tensor_bytes = demand_od.numel() * demand_od.element_size()
print(f"demand_od={tuple(demand_od.shape)}, {tensor_bytes / 2**20:.1f} MiB")
print(f"clean observed trips={int(demand_od.sum()):,}; train route pairs={len(train_routes):,}")


### 5. Build the spatial graph and route priors



In [ ]:
gdf = gpd.read_file(SHAPEFILE)
gdf = gdf[gdf["LocationID"].between(1, CFG.n_zones)].sort_values("LocationID").reset_index(drop=True)
expected_ids = np.arange(1, CFG.n_zones + 1)
if len(gdf) != CFG.n_zones or not np.array_equal(gdf.LocationID.to_numpy(), expected_ids):
    raise ValueError("Taxi-zone IDs are not exactly 1..263 after filtering.")

projected = gdf.to_crs(2263)
centroids = projected.geometry.centroid
xy = torch.tensor(np.c_[centroids.x, centroids.y], dtype=torch.float32)
distance_miles = torch.cdist(xy, xy) / 5280.0
joined = gpd.sjoin(
    projected[["LocationID", "geometry"]],
    projected[["LocationID", "geometry"]],
    how="inner",
    predicate="touches",
)
adjacency = torch.zeros((CFG.n_zones, CFG.n_zones), dtype=torch.bool)
adjacency[joined.index.to_numpy(), joined["index_right"].to_numpy()] = True
nearest = torch.topk(
    distance_miles.masked_fill(torch.eye(CFG.n_zones, dtype=torch.bool), float("inf")),
    k=CFG.nearest_neighbors,
    largest=False,
).indices
src = torch.arange(CFG.n_zones)[:, None].expand_as(nearest)
adjacency[src, nearest] = True
adjacency = adjacency | adjacency.T
adjacency.fill_diagonal_(False)

degree = adjacency.sum(1)
max_actions = int(degree.max()) + 1
neighbor_index = torch.full((CFG.n_zones, max_actions), -1, dtype=torch.long)
valid_action = torch.zeros_like(neighbor_index, dtype=torch.bool)
for zone in range(CFG.n_zones):
    destinations = torch.where(adjacency[zone])[0]
    neighbor_index[zone, 0] = zone
    neighbor_index[zone, 1 : len(destinations) + 1] = destinations
    valid_action[zone, : len(destinations) + 1] = True

a = adjacency.to(torch.float32) + torch.eye(CFG.n_zones)
inv_sqrt_degree = a.sum(1).pow(-0.5)
adj_norm = inv_sqrt_degree[:, None] * a * inv_sqrt_degree[None, :]

global_receipts = float(train_routes.receipts.median())
duration_minutes = (3.0 + distance_miles / (12.0 / 60.0)).clamp(3.0, 120.0)
expected_receipts = torch.full((CFG.n_zones, CFG.n_zones), global_receipts)
ri = torch.as_tensor(train_routes.pu.to_numpy(), dtype=torch.long)
rj = torch.as_tensor(train_routes["dropoff_zone"].to_numpy(), dtype=torch.long)
rn = torch.as_tensor(train_routes.n.to_numpy(), dtype=torch.float32)
rd = torch.as_tensor(train_routes.duration_min.to_numpy(), dtype=torch.float32)
rr = torch.as_tensor(train_routes.receipts.to_numpy(), dtype=torch.float32)
duration_minutes[ri, rj] = rd
expected_receipts[ri, rj] = (
    rn * rr + CFG.fare_prior_strength * global_receipts
) / (rn + CFG.fare_prior_strength)
travel_bins = torch.ceil(duration_minutes / CFG.bin_minutes).to(torch.long).clamp_min(1)

print(f"edges={int(adjacency.sum() // 2)}; degree={int(degree.min())}..{int(degree.max())}")
print(f"actions={max_actions}; training-only median receipts USD {global_receipts:.2f}")


### 6. Build causal features and chronological month splits



In [ ]:
bins = torch.arange(CFG.total_bins)
bin_of_day = bins % CFG.bins_per_day
day = bins // CFG.bins_per_day
day_of_week = day % 7
group = day_of_week * CFG.bins_per_day + bin_of_day
profile_flat = torch.zeros((7 * CFG.bins_per_day, CFG.n_zones))
profile_flat.index_add_(0, group[: CFG.train_end], demand_zone[: CFG.train_end])
group_count = torch.bincount(group[: CFG.train_end], minlength=7 * CFG.bins_per_day).clamp_min(1)
profile_flat /= group_count[:, None]
forecast_zone = profile_flat[group]
train_scale = torch.quantile(demand_zone[: CFG.train_end], 0.95, dim=0).clamp_min(1.0)
pickup_share = demand_zone[: CFG.train_end].sum(0)
pickup_share /= pickup_share.sum()


def integer_partition(weights: torch.Tensor, totals: torch.Tensor) -> torch.Tensor:
    '''Largest-remainder allocation with exact integer row totals.'''
    weights = weights.clamp_min(0)
    probs = weights / weights.sum(1, keepdim=True).clamp_min(1e-12)
    raw = probs * totals[:, None].to(probs.dtype)
    base = torch.floor(raw).to(torch.long)
    remainder = totals.to(torch.long) - base.sum(1)
    frac = raw - base
    order = frac.argsort(dim=1, descending=True)
    rank = torch.empty_like(order)
    rank_source = torch.arange(order.shape[1], device=order.device)[None, :].expand_as(order)
    rank.scatter_(1, order, rank_source)
    return base + (rank < remainder[:, None]).to(torch.long)


initial_idle = integer_partition(pickup_share[None, :], torch.tensor([CFG.fleet_size])).squeeze(0)
hour_angle = 2 * math.pi * bin_of_day / CFG.bins_per_day
dow_angle = 2 * math.pi * day_of_week / 7
time_features = torch.stack(
    (hour_angle.sin(), hour_angle.cos(), dow_angle.sin(), dow_angle.cos()), dim=1
)
split_starts = {
    "train": torch.arange(0, CFG.train_end, CFG.bins_per_day),
    "validation": torch.arange(CFG.train_end, CFG.val_end, CFG.bins_per_day),
    "test": torch.arange(CFG.val_end, CFG.total_bins, CFG.bins_per_day),
}
assert int(initial_idle.sum()) == CFG.fleet_size
assert set(split_starts["train"].tolist()).isdisjoint(split_starts["test"].tolist())
print({name: starts.tolist() for name, starts in split_starts.items()})


## 3. Simulator



In [ ]:
class TorchTaxiFleetEnv(gym.Env):
    '''Gym-compatible simulator with tensor-native reset_tensor and step_tensor methods.'''

    metadata = {"render_modes": []}

    def __init__(
        self,
        cfg: Config,
        demand_od: torch.Tensor,
        demand_zone: torch.Tensor,
        forecast_zone: torch.Tensor,
        time_features: torch.Tensor,
        train_scale: torch.Tensor,
        initial_idle: torch.Tensor,
        duration_minutes: torch.Tensor,
        travel_bins: torch.Tensor,
        expected_receipts: torch.Tensor,
        neighbor_index: torch.Tensor,
        valid_action: torch.Tensor,
        episode_starts: torch.Tensor,
        device: torch.device,
    ) -> None:
        super().__init__()
        self.cfg, self.device = cfg, device
        self.n, self.a = cfg.n_zones, neighbor_index.shape[1]
        tensors = {
            "demand_od": demand_od,
            "demand_zone": demand_zone,
            "forecast_zone": forecast_zone,
            "time_features": time_features,
            "train_scale": train_scale,
            "initial_idle": initial_idle,
            "duration_minutes": duration_minutes,
            "travel_bins": travel_bins,
            "expected_receipts": expected_receipts,
            "neighbor_index": neighbor_index,
            "valid_action": valid_action,
            "episode_starts": episode_starts,
        }
        for name, tensor in tensors.items():
            # Keep the full-year OD cube on host memory; only the active
            # half-hour slice is copied to GPU in step_flows_tensor.
            setattr(
                self,
                name,
                tensor.contiguous() if name == "demand_od" else tensor.to(device),
            )
        self.max_delay = int(travel_bins.max())
        self.ring_size = self.max_delay + 1
        self._origin_grid = torch.arange(self.n, device=device)[:, None]
        self._destination_grid = torch.arange(self.n, device=device)[None, :]
        self._generator = torch.Generator(device=device).manual_seed(cfg.seed)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.n, 8), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.n, self.a), dtype=np.float32
        )

    def reset_tensor(
        self, seed: int | None = None, start_bin: int | None = None
    ) -> torch.Tensor:
        if seed is not None:
            self._generator.manual_seed(seed)
        if start_bin is None:
            ix = torch.randint(
                len(self.episode_starts),
                (1,),
                generator=self._generator,
                device=self.device,
            )
            start_bin = int(self.episode_starts[ix].item())
        self.t, self.steps, self.ring_pos = int(start_bin), 0, 0
        self.idle = self.initial_idle.clone()
        self.occupied_arrivals = torch.zeros(
            (self.ring_size, self.n), dtype=torch.long, device=self.device
        )
        self.reposition_arrivals = torch.zeros_like(self.occupied_arrivals)
        self._assert_conservation()
        return self._observation()

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        super().reset(seed=seed)
        start_bin = None if options is None else options.get("start_bin")
        obs = self.reset_tensor(seed=seed, start_bin=start_bin)
        return obs.detach().cpu().numpy(), {}

    def _observation(self) -> torch.Tensor:
        incoming = (
            self.occupied_arrivals[(self.ring_pos + 1) % self.ring_size]
            + self.reposition_arrivals[(self.ring_pos + 1) % self.ring_size]
        )
        return torch.cat(
            (
                (self.idle / self.cfg.fleet_size).unsqueeze(1),
                (self.demand_zone[self.t] / self.train_scale).unsqueeze(1),
                (self.forecast_zone[self.t] / self.train_scale).unsqueeze(1),
                (incoming / self.cfg.fleet_size).unsqueeze(1),
                # Flatten defensively so both [F] and [1, F] feature layouts
                # broadcast to one feature row per zone.
                self.time_features[self.t].reshape(1, -1).expand(self.n, -1),
            ),
            dim=1,
        ).float()

    @staticmethod
    def _allocate_requests(
        capacity: torch.Tensor, totals: torch.Tensor
    ) -> torch.Tensor:
        totals = torch.minimum(totals.long(), capacity.sum(1))
        allocation = integer_partition(capacity.float(), totals)
        if torch.any(allocation > capacity):
            raise RuntimeError("Request allocation exceeded observed OD capacity.")
        return allocation

    def _schedule(
        self, ring: torch.Tensor, flows: torch.Tensor, delays: torch.Tensor
    ) -> None:
        positive = flows > 0
        if torch.any(positive):
            destination = self._destination_grid.expand_as(flows)[positive]
            slot = (self.ring_pos + delays[positive]) % self.ring_size
            ring.view(-1).scatter_add_(
                0, slot * self.n + destination, flows[positive]
            )

    def logits_to_flows(self, logits: torch.Tensor) -> torch.Tensor:
        logits = logits.to(self.device, dtype=torch.float32)
        if logits.shape != (self.n, self.a):
            raise ValueError(
                f"Expected action {(self.n, self.a)}, got {tuple(logits.shape)}"
            )
        probs = torch.softmax(
            logits.masked_fill(~self.valid_action, -1e9), dim=1
        )
        flows = integer_partition(probs, self.idle)
        return self._limit_relocations(flows)

    def _limit_relocations(self, flows: torch.Tensor) -> torch.Tensor:
        '''Cap empty moves per zone and return excess vehicles to stay.'''
        move_total = flows[:, 1:].sum(1)
        limit = torch.floor(
            self.idle.float() * self.cfg.max_relocation_fraction
        ).to(torch.long)
        capped_total = torch.minimum(move_total, limit)
        capped_moves = integer_partition(
            flows[:, 1:].float(), capped_total
        )
        stay = self.idle - capped_moves.sum(1)
        return torch.cat((stay[:, None], capped_moves), dim=1)

    def step_flows_tensor(self, flows: torch.Tensor):
        flows = flows.to(self.device, dtype=torch.long)
        if flows.shape != (self.n, self.a):
            raise ValueError("Wrong flow shape.")
        if torch.any(flows < 0):
            raise ValueError("Flows must be nonnegative.")
        if torch.any(flows.masked_select(~self.valid_action) != 0):
            raise ValueError("Flows on padded actions must be zero.")
        flows = self._limit_relocations(flows)
        if not torch.equal(flows.sum(1), self.idle):
            raise ValueError(
                "Each zone must allocate all idle vehicles, including stay."
            )

        self.idle.zero_()
        self.idle += flows[:, 0]
        move_flows = flows[:, 1:]
        move_dest = self.neighbor_index[:, 1:].clamp_min(0)
        move_src = self._origin_grid.expand_as(move_dest)
        move_valid = self.valid_action[:, 1:]
        move_delay = self.travel_bins[move_src, move_dest]
        move_minutes = self.duration_minutes[move_src, move_dest]
        flat_index = (
            (self.ring_pos + move_delay) % self.ring_size
        ) * self.n + move_dest
        self.reposition_arrivals.view(-1).scatter_add_(
            0, flat_index[move_valid], move_flows[move_valid]
        )
        relocation_minutes = (
            move_flows[move_valid] * move_minutes[move_valid]
        ).sum()

        od_requests = self.demand_od[self.t].to(
            self.device, dtype=torch.long, non_blocking=True
        )
        requests_by_origin = od_requests.sum(1)
        matched_by_origin = torch.minimum(self.idle, requests_by_origin)
        matched_od = self._allocate_requests(od_requests, matched_by_origin)
        self.idle -= matched_by_origin
        self._schedule(self.occupied_arrivals, matched_od, self.travel_bins)

        served, requested = matched_od.sum(), od_requests.sum()
        missed = requested - served
        gross = (matched_od * self.expected_receipts).sum()
        relocation_cost = (
            relocation_minutes * self.cfg.relocation_cost_per_minute
        )
        objective = (
            gross
            - relocation_cost
            - missed * self.cfg.missed_request_penalty
        )
        service_rate = served.float() / requested.float().clamp_min(1.0)
        reward = (
            objective / self.cfg.fleet_size
            + self.cfg.service_rate_reward * service_rate
        )
        occupied = self.occupied_arrivals.sum()
        repositioning = self.reposition_arrivals.sum()

        self.steps += 1
        terminated = self.steps >= self.cfg.bins_per_day
        self.ring_pos = (self.ring_pos + 1) % self.ring_size
        self.t += 1
        if not terminated:
            self.idle += self.occupied_arrivals[self.ring_pos]
            self.idle += self.reposition_arrivals[self.ring_pos]
            self.occupied_arrivals[self.ring_pos].zero_()
            self.reposition_arrivals[self.ring_pos].zero_()
        self._assert_conservation()

        info = {
            "requested": requested.item(),
            "served": served.item(),
            "missed": missed.item(),
            "gross_receipts": gross.item(),
            "relocation_minutes": relocation_minutes.item(),
            "relocation_cost": relocation_cost.item(),
            "objective": objective.item(),
            "paid_driver_bins": occupied.item(),
            "reposition_driver_bins": repositioning.item(),
            "served_fraction": service_rate.item(),
            "stay_vehicles": flows[:, 0].sum().item(),
            "relocated_vehicles": move_flows.sum().item(),
            "relocation_fraction": (
                move_flows.sum().float() / self.cfg.fleet_size
            ).item(),
        }
        obs = None if terminated else self._observation()
        return obs, reward.float(), terminated, False, info

    def step_tensor(self, action_logits: torch.Tensor):
        return self.step_flows_tensor(
            self.logits_to_flows(action_logits)
        )

    def step(self, action: np.ndarray):
        obs, reward, terminated, truncated, info = self.step_tensor(
            torch.as_tensor(action, device=self.device)
        )
        obs_np = np.zeros(self.observation_space.shape, np.float32)
        if obs is not None:
            obs_np = obs.detach().cpu().numpy()
        return obs_np, reward.item(), terminated, truncated, info

    def _assert_conservation(self) -> None:
        total = (
            self.idle.sum()
            + self.occupied_arrivals.sum()
            + self.reposition_arrivals.sum()
        )
        if int(total) != self.cfg.fleet_size:
            raise RuntimeError(
                f"Vehicle conservation failed: {int(total)}"
            )


def make_env(
    split: str, device: torch.device = DEVICE
) -> TorchTaxiFleetEnv:
    return TorchTaxiFleetEnv(
        CFG,
        demand_od,
        demand_zone,
        forecast_zone,
        time_features,
        train_scale,
        initial_idle,
        duration_minutes,
        travel_bins,
        expected_receipts,
        neighbor_index,
        valid_action,
        split_starts[split],
        device,
    )


smoke_env = make_env("train")
smoke_env.reset_tensor(start_bin=int(split_starts["train"][0]))
stay = torch.zeros(
    (CFG.n_zones, max_actions), dtype=torch.long, device=DEVICE
)
totals = {
    "requested": 0.0,
    "served": 0.0,
    "missed": 0.0,
    "relocation_minutes": 0.0,
}
for _ in range(CFG.bins_per_day):
    stay.zero_()
    stay[:, 0] = smoke_env.idle
    _, _, done, _, info = smoke_env.step_flows_tensor(stay)
    for key in totals:
        totals[key] += info[key]
assert done
assert totals["served"] + totals["missed"] == totals["requested"]
assert totals["relocation_minutes"] == 0.0
print(totals)
print("environment invariants passed")
del smoke_env
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 4. Policy, critic, and PPO



In [ ]:
class GraphBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.self_linear = nn.Linear(in_dim, out_dim)
        self.neighbor_linear = nn.Linear(
            in_dim, out_dim, bias=False
        )
        self.norm = nn.LayerNorm(out_dim)

    def forward(
        self, x: torch.Tensor, adj: torch.Tensor
    ) -> torch.Tensor:
        aggregated = torch.matmul(adj, x)
        return F.silu(
            self.norm(
                self.self_linear(x)
                + self.neighbor_linear(aggregated)
            )
        )


class GraphPolicy(nn.Module):
    def __init__(
        self, obs_dim: int, hidden: int, action_dim: int
    ) -> None:
        super().__init__()
        self.g1 = GraphBlock(obs_dim, hidden)
        self.g2 = GraphBlock(hidden, hidden)
        self.mean_head = nn.Linear(hidden, action_dim)
        self.log_std = nn.Parameter(
            torch.full((1, 1, action_dim), -0.7)
        )

    def distribution(
        self,
        obs: torch.Tensor,
        adj: torch.Tensor,
        valid: torch.Tensor,
    ):
        unbatched = obs.ndim == 2
        x = obs.unsqueeze(0) if unbatched else obs
        h = self.g2(self.g1(x, adj), adj)
        mean = self.mean_head(h).masked_fill(
            ~valid.unsqueeze(0), 0.0
        )
        std = (
            self.log_std.clamp(-3.0, 0.5)
            .exp()
            .expand_as(mean)
        )
        if unbatched:
            mean, std = mean[0], std[0]
        return torch.distributions.Normal(mean, std)

    @staticmethod
    def zone_log_prob(
        dist,
        action: torch.Tensor,
        valid: torch.Tensor,
    ) -> torch.Tensor:
        mask = valid if action.ndim == 2 else valid.unsqueeze(0)
        return (dist.log_prob(action) * mask).sum(-1)

    @staticmethod
    def zone_entropy(
        dist, valid: torch.Tensor
    ) -> torch.Tensor:
        entropy = dist.entropy()
        mask = (
            valid
            if entropy.ndim == 2
            else valid.unsqueeze(0)
        )
        return (entropy * mask).sum(-1)


class CentralCritic(nn.Module):
    def __init__(self, obs_dim: int, hidden: int) -> None:
        super().__init__()
        self.g1 = GraphBlock(obs_dim, hidden)
        self.g2 = GraphBlock(hidden, hidden)
        self.value = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def forward(
        self, obs: torch.Tensor, adj: torch.Tensor
    ) -> torch.Tensor:
        unbatched = obs.ndim == 2
        x = obs.unsqueeze(0) if unbatched else obs
        h = self.g2(self.g1(x, adj), adj)
        pooled = torch.cat(
            (h.mean(1), h.amax(1)), dim=-1
        )
        value = self.value(pooled).squeeze(-1)
        return value[0] if unbatched else value


@dataclass
class Rollout:
    states: list[torch.Tensor]
    actions: list[torch.Tensor]
    old_logp: list[torch.Tensor]
    rewards: list[torch.Tensor]
    values: list[torch.Tensor]
    next_values: list[torch.Tensor]
    dones: list[bool]

    @classmethod
    def empty(cls):
        return cls([], [], [], [], [], [], [])

    def tensors(
        self, cfg: Config
    ) -> dict[str, torch.Tensor]:
        data = {
            "state": torch.stack(self.states),
            "action": torch.stack(self.actions),
            "old_logp": torch.stack(self.old_logp),
            "reward": torch.stack(self.rewards),
            "old_value": torch.stack(self.values),
            "next_value": torch.stack(self.next_values),
            "done": torch.tensor(
                self.dones,
                dtype=torch.float32,
                device=self.values[0].device,
            ),
        }
        advantage = torch.zeros_like(data["reward"])
        gae = torch.zeros_like(data["reward"][0])
        for t in reversed(range(len(self.rewards))):
            mask = 1.0 - data["done"][t]
            delta = (
                data["reward"][t]
                + cfg.gamma * data["next_value"][t] * mask
                - data["old_value"][t]
            )
            gae = (
                delta
                + cfg.gamma * cfg.gae_lambda * mask * gae
            )
            advantage[t] = gae
        data["returns"] = advantage + data["old_value"]
        data["advantage"] = (
            advantage - advantage.mean()
        ) / advantage.std().clamp_min(1e-6)
        return data


def collect_rollout(
    env,
    policy,
    critic,
    adj,
    valid,
    n_days: int,
    seed: int,
) -> Rollout:
    rollout = Rollout.empty()
    policy.eval()
    critic.eval()
    for day_ix in range(n_days):
        obs = env.reset_tensor(seed=seed + day_ix)
        for _ in range(env.cfg.bins_per_day):
            with torch.no_grad():
                dist = policy.distribution(
                    obs, adj, valid
                )
                action = dist.sample()
                zone_logp = policy.zone_log_prob(
                    dist, action, valid
                )
                value = critic(obs, adj)
                next_obs, reward, done, _, _ = (
                    env.step_tensor(action)
                )
                next_value = (
                    torch.zeros_like(value)
                    if done
                    else critic(next_obs, adj)
                )
            rollout.states.append(obs)
            rollout.actions.append(action)
            rollout.old_logp.append(zone_logp)
            rollout.rewards.append(reward)
            rollout.values.append(value)
            rollout.next_values.append(next_value)
            rollout.dones.append(done)
            if done:
                break
            obs = next_obs
    return rollout


def ppo_update(
    policy,
    critic,
    actor_opt,
    critic_opt,
    data,
    adj,
    valid,
    cfg,
) -> dict[str, float]:
    policy.train()
    critic.train()
    logs = {
        "actor_loss": [],
        "critic_loss": [],
        "entropy": [],
        "approx_kl": [],
    }
    n_steps = data["state"].shape[0]
    for _ in range(cfg.ppo_epochs):
        order = torch.randperm(
            n_steps, device=data["state"].device
        )
        for start in range(
            0, n_steps, cfg.minibatch_steps
        ):
            ix = order[
                start : start + cfg.minibatch_steps
            ]
            dist = policy.distribution(
                data["state"][ix], adj, valid
            )
            new_logp = policy.zone_log_prob(
                dist, data["action"][ix], valid
            )
            log_ratio = (
                new_logp - data["old_logp"][ix]
            ).clamp(-20, 20)
            ratio = log_ratio.exp()
            advantage = data["advantage"][ix, None]
            surrogate = torch.minimum(
                ratio * advantage,
                ratio.clamp(
                    1 - cfg.clip_eps,
                    1 + cfg.clip_eps,
                )
                * advantage,
            )
            entropy = policy.zone_entropy(
                dist, valid
            ).mean()
            actor_loss = (
                -surrogate.mean()
                - cfg.entropy_coef * entropy
            )
            actor_opt.zero_grad(set_to_none=True)
            actor_loss.backward()
            nn.utils.clip_grad_norm_(
                policy.parameters(), cfg.grad_clip
            )
            actor_opt.step()

            value = critic(data["state"][ix], adj)
            old_value = data["old_value"][ix]
            target = data["returns"][ix]
            clipped_value = old_value + (
                value - old_value
            ).clamp(-cfg.clip_eps, cfg.clip_eps)
            value_loss = 0.5 * torch.maximum(
                (value - target).square(),
                (clipped_value - target).square(),
            ).mean()
            critic_loss = cfg.value_coef * value_loss
            critic_opt.zero_grad(set_to_none=True)
            critic_loss.backward()
            nn.utils.clip_grad_norm_(
                critic.parameters(), cfg.grad_clip
            )
            critic_opt.step()

            logs["actor_loss"].append(
                float(actor_loss.detach())
            )
            logs["critic_loss"].append(
                float(critic_loss.detach())
            )
            logs["entropy"].append(
                float(entropy.detach())
            )
            logs["approx_kl"].append(
                float(
                    (
                        (ratio - 1) - log_ratio
                    ).mean().detach()
                )
            )
    return {
        key: float(np.mean(values))
        for key, values in logs.items()
    }


## 5. Training and validation



In [ ]:
def summarize_episode(
    rows: list[dict], cfg: Config
) -> dict[str, float]:
    totals = pd.DataFrame(rows).sum(numeric_only=True)
    requested, steps = totals["requested"], len(rows)
    return {
        "objective": totals["objective"],
        "observed_service_rate": (
            totals["served"] / max(requested, 1.0)
        ),
        "gross_receipts": totals["gross_receipts"],
        "missed": totals["missed"],
        "relocation_minutes": totals["relocation_minutes"],
        "paid_utilization": (
            totals["paid_driver_bins"]
            / (cfg.fleet_size * steps)
        ),
        "reposition_utilization": (
            totals["reposition_driver_bins"]
            / (cfg.fleet_size * steps)
        ),
        "relocated_vehicles_mean": (
            totals.get("relocated_vehicles", 0.0) / steps
        ),
        "relocation_fraction_mean": (
            totals.get("relocation_fraction", 0.0) / steps
        ),
    }


@torch.no_grad()
def evaluate_learned(
    env,
    policy,
    adj,
    valid,
    starts: Iterable[int],
) -> pd.DataFrame:
    policy.eval()
    output = []
    for start in starts:
        obs = env.reset_tensor(start_bin=int(start))
        rows = []
        for _ in range(env.cfg.bins_per_day):
            action = policy.distribution(
                obs, adj, valid
            ).mean
            obs, _, done, _, info = env.step_tensor(
                action
            )
            rows.append(info)
            if done:
                break
        output.append(
            {
                "start_bin": int(start),
                **summarize_episode(rows, env.cfg),
            }
        )
    return pd.DataFrame(output)


def train_one(seed: int):
    seed_everything(seed)
    train_env = make_env("train")
    val_env = make_env("validation")
    adj = adj_norm.to(DEVICE)
    valid = valid_action.to(DEVICE)
    policy = GraphPolicy(
        8, CFG.hidden_dim, max_actions
    ).to(DEVICE)
    # Start from a conservative no-rebalancing prior.
    with torch.no_grad():
        policy.mean_head.bias.zero_()
        policy.mean_head.bias[0] = CFG.stay_logit_bias
    critic = CentralCritic(
        8, CFG.hidden_dim
    ).to(DEVICE)
    actor_opt = torch.optim.Adam(
        policy.parameters(), lr=CFG.actor_lr, eps=1e-5
    )
    critic_opt = torch.optim.Adam(
        critic.parameters(), lr=CFG.critic_lr, eps=1e-5
    )
    best_score = -float("inf")
    best = None
    stale = 0
    history = []
    val_starts = (
        val_env.episode_starts.detach().cpu().tolist()
    )

    for update in range(1, CFG.max_updates + 1):
        rollout = collect_rollout(
            train_env,
            policy,
            critic,
            adj,
            valid,
            CFG.rollout_days,
            seed + 10_000 * update,
        )
        stats = ppo_update(
            policy,
            critic,
            actor_opt,
            critic_opt,
            rollout.tensors(CFG),
            adj,
            valid,
            CFG,
        )
        stats["update"] = update
        if update % CFG.validation_every == 0:
            val = evaluate_learned(
                val_env,
                policy,
                adj,
                valid,
                val_starts,
            )
            score = float(val["objective"].mean())
            stats["validation_objective"] = score
            if score > best_score:
                best_score, stale = score, 0
                best = {
                    "policy": copy.deepcopy(
                        policy.state_dict()
                    ),
                    "critic": copy.deepcopy(
                        critic.state_dict()
                    ),
                    "update": update,
                    "validation_objective": score,
                }
            else:
                stale += 1
        history.append(stats)
        if (
            update == 1
            or update % CFG.validation_every == 0
        ):
            print(
                f"seed={seed} update={update:03d} "
                f"actor={stats['actor_loss']:.3f} "
                f"critic={stats['critic_loss']:.3f} "
                f"val={stats.get('validation_objective', float('nan')):.1f}"
            )
        if stale >= CFG.early_stop_checks:
            break

    if best is None:
        val = evaluate_learned(
            val_env,
            policy,
            adj,
            valid,
            val_starts,
        )
        best = {
            "policy": copy.deepcopy(policy.state_dict()),
            "critic": copy.deepcopy(critic.state_dict()),
            "update": len(history),
            "validation_objective": float(
                val["objective"].mean()
            ),
        }
    policy.load_state_dict(best["policy"])
    critic.load_state_dict(best["critic"])
    return (
        policy,
        critic,
        pd.DataFrame(history),
        best,
    )


trained = {
    seed: train_one(seed)
    for seed in CFG.training_seeds
}


## 6. Baselines and locked test evaluation



In [ ]:
def no_rebalance_flows(
    env: TorchTaxiFleetEnv,
) -> torch.Tensor:
    flows = torch.zeros(
        (env.n, env.a),
        dtype=torch.long,
        device=env.device,
    )
    flows[:, 0] = env.idle
    return flows


def greedy_flows(
    env: TorchTaxiFleetEnv,
) -> torch.Tensor:
    destination = env.neighbor_index.clamp_min(0)
    forecast = env.forecast_zone[
        min(env.t + 1, env.cfg.total_bins - 1)
    ][destination]
    travel = env.duration_minutes[
        torch.arange(env.n, device=env.device)[:, None],
        destination,
    ]
    score = (
        (forecast + 1.0) / (1.0 + 0.05 * travel)
    ).masked_fill(~env.valid_action, -float("inf"))
    best = score.argmax(1)
    flows = torch.zeros(
        (env.n, env.a),
        dtype=torch.long,
        device=env.device,
    )
    flows.scatter_(1, best[:, None], env.idle[:, None])
    return flows


def rolling_milp_flows(
    env: TorchTaxiFleetEnv,
) -> torch.Tensor:
    '''One-bin target MILP baseline, not an oracle.'''
    valid = env.valid_action.detach().cpu().numpy()
    origins, slots = np.where(valid)
    destinations = (
        env.neighbor_index.detach().cpu().numpy()[
            origins, slots
        ]
    )
    idle = env.idle.detach().cpu().numpy().astype(float)
    target = (
        env.forecast_zone[
            min(env.t + 1, env.cfg.total_bins - 1)
        ]
        .detach()
        .cpu()
        .numpy()
    )
    edge_cost = np.where(
        origins == destinations,
        0.0,
        env.duration_minutes.detach().cpu().numpy()[
            origins, destinations
        ]
        * env.cfg.relocation_cost_per_minute,
    )
    n_edges = len(origins)
    c = np.r_[
        edge_cost,
        np.full(
            env.n, env.cfg.missed_request_penalty
        ),
    ]
    by_origin = sparse.coo_matrix(
        (
            np.ones(n_edges),
            (origins, np.arange(n_edges)),
        ),
        shape=(env.n, n_edges),
    )
    by_destination = sparse.coo_matrix(
        (
            np.ones(n_edges),
            (destinations, np.arange(n_edges)),
        ),
        shape=(env.n, n_edges),
    )
    constraints = [
        LinearConstraint(
            sparse.hstack(
                (
                    by_origin,
                    sparse.csr_matrix(
                        (env.n, env.n)
                    ),
                )
            ),
            idle,
            idle,
        ),
        LinearConstraint(
            sparse.hstack(
                (by_destination, sparse.eye(env.n))
            ),
            target,
            np.full(env.n, np.inf),
        ),
    ]
    upper = np.r_[
        idle[origins], np.full(env.n, np.inf)
    ]
    result = milp(
        c,
        integrality=np.r_[
            np.ones(n_edges), np.zeros(env.n)
        ],
        bounds=Bounds(
            np.zeros_like(upper), upper
        ),
        constraints=constraints,
        options={
            "time_limit": 20.0,
            "mip_rel_gap": 0.02,
        },
    )
    if not result.success:
        return greedy_flows(env)
    weights = torch.zeros(
        (env.n, env.a), device=env.device
    )
    weights[
        torch.as_tensor(
            origins, device=env.device
        ),
        torch.as_tensor(slots, device=env.device),
    ] = torch.as_tensor(
        result.x[:n_edges],
        dtype=torch.float32,
        device=env.device,
    )
    return integer_partition(weights, env.idle)


@torch.no_grad()
def evaluate_baseline(
    name: str,
    policy: Callable,
    starts: Iterable[int],
) -> pd.DataFrame:
    env = make_env("test")
    output = []
    for start in starts:
        env.reset_tensor(start_bin=int(start))
        rows = []
        for _ in range(env.cfg.bins_per_day):
            _, _, done, _, info = (
                env.step_flows_tensor(policy(env))
            )
            rows.append(info)
            if done:
                break
        output.append(
            {
                "method": name,
                "seed": -1,
                "start_bin": int(start),
                **summarize_episode(rows, env.cfg),
            }
        )
    return pd.DataFrame(output)


test_starts = split_starts["test"].tolist()
results = [
    evaluate_baseline(
        "no_rebalancing",
        no_rebalance_flows,
        test_starts,
    ),
    evaluate_baseline(
        "greedy_forecast",
        greedy_flows,
        test_starts,
    ),
]
if not CFG.quick_mode:
    results.append(
        evaluate_baseline(
            "rolling_milp",
            rolling_milp_flows,
            test_starts,
        )
    )
for seed, (
    policy,
    _,
    _,
    _,
) in trained.items():
    learned = evaluate_learned(
        make_env("test"),
        policy,
        adj_norm.to(DEVICE),
        valid_action.to(DEVICE),
        test_starts,
    )
    learned.insert(0, "seed", seed)
    learned.insert(0, "method", "mappo_gcn")
    results.append(learned)
test_results = pd.concat(results, ignore_index=True)
test_results


## 7. Uncertainty and plots



In [ ]:
def paired_bootstrap(
    frame,
    method,
    baseline,
    metric,
    draws=10_000,
    seed=2026,
):
    left = (
        frame[frame.method == method]
        .groupby("start_bin")[metric]
        .mean()
    )
    right = (
        frame[frame.method == baseline]
        .groupby("start_bin")[metric]
        .mean()
    )
    paired = pd.concat(
        (left, right), axis=1, join="inner"
    ).dropna()
    delta = (
        paired.iloc[:, 0].to_numpy()
        - paired.iloc[:, 1].to_numpy()
    )
    rng = np.random.default_rng(seed)
    sampled = rng.choice(
        delta,
        size=(draws, len(delta)),
        replace=True,
    ).mean(1)
    lo, hi = np.quantile(
        sampled, [0.025, 0.975]
    )
    return {
        "method": method,
        "baseline": baseline,
        "metric": metric,
        "mean_paired_difference": delta.mean(),
        "bootstrap_2.5%": lo,
        "bootstrap_97.5%": hi,
        "paired_days": len(delta),
    }


summary = (
    test_results.groupby("method")
    .agg(
        objective_mean=("objective", "mean"),
        objective_sd=("objective", "std"),
        service_rate_mean=(
            "observed_service_rate", "mean"
        ),
        gross_receipts_mean=(
            "gross_receipts", "mean"
        ),
        missed_mean=("missed", "mean"),
        relocation_minutes_mean=(
            "relocation_minutes", "mean"
        ),
        paid_utilization_mean=(
            "paid_utilization", "mean"
        ),
        reposition_utilization_mean=(
            "reposition_utilization", "mean"
        ),
    )
    .sort_values(
        "objective_mean", ascending=False
    )
)
display(summary)
comparisons = pd.DataFrame(
    [
        paired_bootstrap(
            test_results,
            "mappo_gcn",
            baseline,
            metric,
        )
        for baseline in (
            "no_rebalancing",
            "greedy_forecast",
        )
        for metric in (
            "objective",
            "observed_service_rate",
        )
    ]
)
display(comparisons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
test_results.boxplot(
    column="objective",
    by="method",
    ax=axes[0],
    grid=False,
)
test_results.boxplot(
    column="observed_service_rate",
    by="method",
    ax=axes[1],
    grid=False,
)
axes[0].set_title("Daily simulation objective")
axes[1].set_title(
    "Observed-trip service rate"
)
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("")
fig.tight_layout()
plt.show()
if CFG.quick_mode:
    print(
        "QUICK MODE is a pipeline check only. "
        "Set quick_mode=False and rerun for the plan."
    )


## 8. GPU/CPU benchmark and artifact export



In [ ]:
@torch.no_grad()
def benchmark_policy(
    policy: GraphPolicy,
    device: torch.device,
    repeats: int = 100,
) -> float:
    model = copy.deepcopy(policy).to(device).eval()
    local_adj = adj_norm.to(device)
    local_valid = valid_action.to(device)
    sample = torch.zeros(
        (CFG.n_zones, 8), device=device
    )
    for _ in range(10):
        _ = model.distribution(
            sample, local_adj, local_valid
        ).mean
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.distribution(
            sample, local_adj, local_valid
        ).mean
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return (
        1e3
        * (time.perf_counter() - start)
        / repeats
    )


representative = next(iter(trained.values()))[0]
timings = {
    "cpu_ms": benchmark_policy(
        representative, torch.device("cpu")
    )
}
if torch.cuda.is_available():
    timings["cuda_ms"] = benchmark_policy(
        representative, torch.device("cuda")
    )
print(timings)
print(
    "This 263-node model is small, so GPU launch "
    "overhead can dominate. Choose hardware from "
    "synchronized full-rollout time, not availability."
)


@torch.no_grad()
def benchmark_day(
    policy: GraphPolicy,
    device: torch.device,
) -> dict[str, float]:
    env = make_env("train", device=device)
    model = copy.deepcopy(policy).to(device).eval()
    local_adj = adj_norm.to(device)
    local_valid = valid_action.to(device)
    obs = env.reset_tensor(
        start_bin=int(split_starts["train"][0])
    )
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    objective = 0.0
    for _ in range(CFG.bins_per_day):
        action = model.distribution(
            obs, local_adj, local_valid
        ).mean
        obs, _, done, _, info = env.step_tensor(action)
        objective += info["objective"]
        if done:
            break
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    return {
        "milliseconds": (
            1e3 * (time.perf_counter() - started)
        ),
        "objective": objective,
    }


end_to_end = {
    "cpu": benchmark_day(
        representative, torch.device("cpu")
    )
}
if torch.cuda.is_available():
    end_to_end["cuda"] = benchmark_day(
        representative, torch.device("cuda")
    )
    cpu_objective = end_to_end["cpu"]["objective"]
    cuda_objective = end_to_end["cuda"]["objective"]
    relative_gap = abs(cpu_objective - cuda_objective) / max(
        abs(cpu_objective), 1.0
    )
    assert np.isfinite(cpu_objective) and np.isfinite(cuda_objective)
    print(
        f"CPU/GPU objective relative gap={relative_gap:.3%}; "
        "integer allocations can amplify harmless floating-point differences."
    )
print({"steady_state_day": end_to_end})

artifact_dir = DATA_DIR / "artifacts"
artifact_dir.mkdir(exist_ok=True)
test_results.to_csv(
    artifact_dir / "locked_test_results.csv",
    index=False,
)
comparisons.to_csv(
    artifact_dir / "paired_comparisons.csv",
    index=False,
)
summary.to_csv(artifact_dir / "summary.csv")
for seed, (
    policy,
    critic,
    history,
    best,
) in trained.items():
    torch.save(
        {
            "config": asdict(CFG),
            "seed": seed,
            "policy_state_dict": (
                policy.state_dict()
            ),
            "critic_state_dict": (
                critic.state_dict()
            ),
            "best_update": best["update"],
            "best_validation_objective": (
                best["validation_objective"]
            ),
            "adjacency": adjacency,
            "neighbor_index": neighbor_index,
            "valid_action": valid_action,
        },
        artifact_dir
        / f"mappo_gcn_seed_{seed}.pt",
    )
    history.to_csv(
        artifact_dir
        / f"training_history_seed_{seed}.csv",
        index=False,
    )
with (
    artifact_dir / "run_manifest.json"
).open("w") as handle:
    json.dump(
        {
            "config": asdict(CFG),
            "device": str(DEVICE),
            "torch_version": torch.__version__,
            "numpy_version": np.__version__,
            "trip_files": [path.name for path in TRIP_FILES],
            "trip_file_bytes": {
                path.name: path.stat().st_size for path in TRIP_FILES
            },
        },
        handle,
        indent=2,
    )
print(f"saved to {artifact_dir}")


## Interpretation gates, follow-ups, and sources

Do not call the policy better from rising training reward. Advance it only if locked paired comparisons are favorable, conservation passes, results are stable across five full-run seeds, and the conclusion survives cost sensitivity.

Recommended follow-ups:

1. Repeat the locked protocol over multiple seasons and years.
2. Sweep relocation and missed-request costs without retraining.
3. Add seasonal, lag-seven-day, shuffled, and perfect-foresight forecast ablations.
4. Compare the GCN against a parameter-shared MLP to test whether the graph adds value.
5. Calibrate fleet supply using an external driver-availability source.
6. Then consider road-network travel times, parallel environments, or a longer-horizon optimizer.

Evidence boundaries and methods:

- NYC TLC describes pickup/drop-off, location, and fare fields and warns that it did not create the trip data or guarantee accuracy: [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
- PPO follows the clipped surrogate formulation in [Schulman et al. (2017)](https://arxiv.org/abs/1707.06347); GAE follows [Schulman et al. (2015/2018)](https://arxiv.org/abs/1506.02438).
- MAPPO is a candidate, not a guaranteed improvement; see [Yu et al. (2022)](https://arxiv.org/abs/2103.01955).
- Structured ideation, adversarial review, decision logging, and the GPU workflow were informed by Timothy Kassis, Vinayak Agarwal, Yuhuan He, Darshil Patel, and Aubrey M. Brueckner (2026), [Scientific Agent Skills: A Library of Procedural Knowledge for Research Agents](https://doi.org/10.48550/arXiv.2609.00065).
